# Validate Sweden Weekly Patterns Data

Explores and validates the generated `sweden_weekly_patterns_2024_03.parquet` file,
comparing its structure with the US weekly_patterns format for cross-country analysis.

**Key validations:**
1. Schema and column completeness
2. Data quality (coverage, missing values)
3. VISITOR_HOME_CBGS parsing and statistics
4. Category distribution comparison with US data
5. Geographic and temporal coverage

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys

# Add project root to path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

# Paths
SE_WP_FILE = ROOT / "dbs/sweden_weekly_patterns/sweden_weekly_patterns_2024_03.parquet"
US_WP_DIR = ROOT / "dbs/us_foot_traffic/weekly_patterns"

print(f"Sweden data: {SE_WP_FILE}")
print(f"US data dir: {US_WP_DIR}")

Sweden data: /workspace/dbs/sweden_weekly_patterns/sweden_weekly_patterns_2024_03.parquet
US data dir: /workspace/dbs/us_foot_traffic/weekly_patterns


## 1. Load Data

In [2]:
df_se = pd.read_parquet(SE_WP_FILE)
print(f"Records: {len(df_se):,}")
print(f"File size: {SE_WP_FILE.stat().st_size / 1e6:.1f} MB")
df_se.head()

Records: 995,666
File size: 78.0 MB


,ID_STORE,PLACEKEY,LOCATION_NAME,STREET_ADDRESS,CITY,REGION,POSTAL_CODE,LATITUDE,LONGITUDE,TOP_CATEGORY,...,NAICS_CODE,BRAND,VISIT_COUNTS,WEIGHTED_VISIT_COUNTS,VISITOR_COUNTS,VISITOR_HOME_CBGS,VISITOR_HOME_CBGS_WEIGHTED,DATE_RANGE_START,DATE_RANGE_END,ISO_COUNTRY_CODE
0,zzy-222@33k-8zs-73q,zzy-222@33k-8zs-73q,Vaderoarnas Vardshus & Konferens,Storö Väderöarna,Fjällbacka,Västra Götaland County,45740,58.576771,11.067054,Traveler Accommodation,...,721110,,6,14.282640,2,"{""1281C1440"": 1, ""1480C1820"": 5}","{""1281C1440"": 6.79, ""1480C1820"": 7.5}",2024-03-04,2024-03-10,SE
1,zzy-222@33k-8zs-73q,zzy-222@33k-8zs-73q,Vaderoarnas Vardshus & Konferens,Storö Väderöarna,Fjällbacka,Västra Götaland County,45740,58.576771,11.067054,Traveler Accommodation,...,721110,,1,1.672624,1,"{""0186C1260"": 1}","{""0186C1260"": 1.67}",2024-03-11,2024-03-17,SE
2,zzy-222@33k-8zs-73q,zzy-222@33k-8zs-73q,Vaderoarnas Vardshus & Konferens,Storö Väderöarna,Fjällbacka,Västra Götaland County,45740,58.576771,11.067054,Traveler Accommodation,...,721110,,1,2.350318,1,"{""1480C1100"": 1}","{""1480C1100"": 2.35}",2024-03-25,2024-03-31,SE
3,zzy-222@33k-94w-zs5,zzy-222@33k-94w-zs5,Sovalls Camping,"Sövallsvägen 91, Sverige",Grebbestad,Västra Götaland County,NaN,58.687799,11.226336,RV (Recreational Vehicle) Parks and Recreation...,...,721214,,9,57.605505,1,"{""1488C1040"": 9}","{""1488C1040"": 57.61}",2024-03-04,2024-03-10,SE
4,zzy-222@33k-94w-zs5,zzy-222@33k-94w-zs5,Sovalls Camping,"Sövallsvägen 91, Sverige",Grebbestad,Västra Götaland County,NaN,58.687799,11.226336,RV (Recreational Vehicle) Parks and Recreation...,...,721214,,1,37.000000,1,"{""1435A0030"": 1}","{""1435A0030"": 37.0}",2024-03-18,2024-03-24,SE


## 2. Schema Validation

In [3]:
# Column info with coverage
schema_info = pd.DataFrame({
    'dtype': df_se.dtypes,
    'non_null': df_se.notna().sum(),
    'coverage_pct': (df_se.notna().sum() / len(df_se) * 100).round(1)
})
schema_info

,dtype,non_null,coverage_pct
ID_STORE,str,995666,100.0
PLACEKEY,str,995666,100.0
LOCATION_NAME,str,995666,100.0
STREET_ADDRESS,str,995666,100.0
CITY,str,949173,95.3
REGION,str,995660,100.0
POSTAL_CODE,str,824627,82.8
LATITUDE,float64,995666,100.0
LONGITUDE,float64,995666,100.0
TOP_CATEGORY,str,995666,100.0


In [4]:
# Check for expected columns (matching US format)
expected_cols = [
    'ID_STORE', 'PLACEKEY', 'LOCATION_NAME', 'LATITUDE', 'LONGITUDE',
    'TOP_CATEGORY', 'SUB_CATEGORY', 'unified_category',
    'VISIT_COUNTS', 'VISITOR_HOME_CBGS',
    'DATE_RANGE_START', 'DATE_RANGE_END'
]

missing_cols = [c for c in expected_cols if c not in df_se.columns]
print(f"Missing expected columns: {missing_cols if missing_cols else 'None'}")

Missing expected columns: None


## 3. Temporal Coverage

In [5]:
# Weekly summary
weekly_summary = df_se.groupby('DATE_RANGE_START').agg({
    'PLACEKEY': ['count', 'nunique'],
    'VISIT_COUNTS': 'sum',
    'WEIGHTED_VISIT_COUNTS': 'sum'
})
weekly_summary.columns = ['poi_weeks', 'unique_pois', 'total_visits', 'weighted_visits']
weekly_summary

,poi_weeks,unique_pois,total_visits,weighted_visits
DATE_RANGE_START,,,,
2024-02-26,148361,148361,605233,2.239234e+06
2024-03-04,216627,216627,1531336,5.669476e+06
2024-03-11,211628,211628,1457399,5.354076e+06
2024-03-18,210601,210601,1529138,5.592141e+06
2024-03-25,208445,208445,1498100,5.512939e+06
2024-04-01,4,4,4,1.520199e+01


In [6]:
print(f"Total POI-weeks: {len(df_se):,}")
print(f"Total unique POIs: {df_se['PLACEKEY'].nunique():,}")
print(f"Weeks covered: {df_se['DATE_RANGE_START'].nunique()}")
print(f"Date range: {df_se['DATE_RANGE_START'].min()} to {df_se['DATE_RANGE_END'].max()}")

Total POI-weeks: 995,666
Total unique POIs: 349,311
Weeks covered: 6
Date range: 2024-02-26 to 2024-04-07


## 4. VISITOR_HOME_CBGS Validation

In [7]:
def parse_cbgs(val):
    """Parse VISITOR_HOME_CBGS JSON and return dict."""
    if pd.isna(val):
        return {}
    try:
        if isinstance(val, str):
            return json.loads(val)
        return val if isinstance(val, dict) else {}
    except:
        return {}

# Sample CBGS parsing
sample_cbgs = df_se['VISITOR_HOME_CBGS'].iloc[0]
print(f"Raw type: {type(sample_cbgs)}")
print(f"Raw value (first 200 chars): {str(sample_cbgs)[:200]}")
print(f"\nParsed: {parse_cbgs(sample_cbgs)}")

Raw type: <class 'str'>
Raw value (first 200 chars): {"1281C1440": 1, "1480C1820": 5}

Parsed: {'1281C1440': 1, '1480C1820': 5}


In [8]:
# Compute CBGS stats
df_se['n_home_zones'] = df_se['VISITOR_HOME_CBGS'].apply(lambda x: len(parse_cbgs(x)))
df_se['cbgs_total'] = df_se['VISITOR_HOME_CBGS'].apply(lambda x: sum(parse_cbgs(x).values()))

print("Home zone distribution:")
df_se['n_home_zones'].describe()

Home zone distribution:


count    995666.000000
mean          2.786000
std           6.517155
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max         906.000000
Name: n_home_zones, dtype: float64

In [9]:
# Consistency check: CBGS sum should equal VISIT_COUNTS
df_se['cbgs_matches'] = df_se['cbgs_total'] == df_se['VISIT_COUNTS']
match_rate = df_se['cbgs_matches'].mean() * 100
print(f"CBGS sum matches VISIT_COUNTS: {match_rate:.1f}%")

if match_rate < 100:
    mismatches = df_se[~df_se['cbgs_matches']]
    print(f"Mismatches: {len(mismatches):,}")
    display(mismatches[['PLACEKEY', 'VISIT_COUNTS', 'cbgs_total', 'n_home_zones']].head())

CBGS sum matches VISIT_COUNTS: 100.0%


## 5. Category Distribution

In [10]:
# Unified category distribution
unified_stats = df_se.groupby('unified_category').agg({
    'PLACEKEY': 'nunique',
    'VISIT_COUNTS': 'sum',
    'n_home_zones': 'mean'
}).rename(columns={
    'PLACEKEY': 'unique_pois',
    'n_home_zones': 'avg_zones'
})
unified_stats['visit_pct'] = (unified_stats['VISIT_COUNTS'] / unified_stats['VISIT_COUNTS'].sum() * 100).round(1)
unified_stats = unified_stats.sort_values('VISIT_COUNTS', ascending=False)
unified_stats

,unique_pois,VISIT_COUNTS,avg_zones,visit_pct
unified_category,,,,
food_dining,29130,571688,3.266319,21.3
retail_specialty,30438,522991,2.831626,19.4
automotive,22915,357935,2.435522,13.3
entertainment_recreation,14711,355974,3.740087,13.2
personal_services,15320,263036,2.537137,9.8
professional_services,8806,162031,2.712257,6.0
education_higher,6277,129663,2.898831,4.8
accommodation_travel,7501,124493,2.640731,4.6
financial_services,4966,89243,2.771358,3.3


In [11]:
# Unmapped categories
unmapped = df_se[df_se['unified_category'].isna()]
print(f"Unmapped records: {len(unmapped):,} ({len(unmapped)/len(df_se)*100:.1f}%)")

if len(unmapped) > 0:
    print("\nTop unmapped SUB_CATEGORIES:")
    display(unmapped.groupby('SUB_CATEGORY').size().sort_values(ascending=False).head(15))

Unmapped records: 577,946 (58.0%)

Top unmapped SUB_CATEGORIES:


SUB_CATEGORY
Residential Property Managers                                       56682
Other Urban Transit Systems                                         45746
Bus and Other Motor Vehicle Transit Systems                         38260
Landscaping Services                                                37732
Residential Remodelers                                              27486
Child Day Care Services                                             23050
Elementary and Secondary Schools                                    23016
Plumbing, Heating, and Air-Conditioning Contractors                 17141
Hair, Nail, and Skin Care Services                                  16398
Parking Lots and Garages                                            16274
Taxi Service                                                        15011
Electrical Contractors and Other Wiring Installation Contractors    11443
Offices of Physicians (except Mental Health Specialists)            11420
Commercial Screen Printin

In [12]:
# Top SUB_CATEGORIES
sub_stats = df_se.groupby('SUB_CATEGORY').agg({
    'PLACEKEY': 'nunique',
    'VISIT_COUNTS': 'sum',
    'unified_category': 'first'
}).sort_values('VISIT_COUNTS', ascending=False)
sub_stats.head(20)

,PLACEKEY,VISIT_COUNTS,unified_category
SUB_CATEGORY,,,
Elementary and Secondary Schools,6203,470327,NaN
Residential Property Managers,21017,321519,NaN
Full-Service Restaurants,14722,270367,food_dining
Fitness and Recreational Sports Centers,8279,241482,entertainment_recreation
Landscaping Services,13797,227546,NaN
Other Urban Transit Systems,18692,210595,NaN
Bus and Other Motor Vehicle Transit Systems,15719,182753,NaN
Child Day Care Services,7868,178853,NaN
Residential Remodelers,9985,160798,NaN


## 6. Geographic Coverage

In [13]:
# City distribution
if 'CITY' in df_se.columns:
    city_stats = df_se.groupby('CITY').agg({
        'PLACEKEY': 'nunique',
        'VISIT_COUNTS': 'sum'
    }).rename(columns={'PLACEKEY': 'unique_pois'})
    city_stats['visit_pct'] = (city_stats['VISIT_COUNTS'] / city_stats['VISIT_COUNTS'].sum() * 100).round(1)
    city_stats = city_stats.sort_values('VISIT_COUNTS', ascending=False)
    display(city_stats.head(20))

,unique_pois,VISIT_COUNTS,visit_pct
CITY,,,
Stockholm,27278,599255,9.4
Göteborg,13242,301873,4.7
Malmö,10322,234502,3.7
Uppsala,6460,144521,2.3
Linköping,4519,126228,2.0
Gävle,3176,112533,1.8
Norrköping,3695,86779,1.4
Solna,2717,86597,1.4
Västerås,4191,80850,1.3


In [14]:
# Coordinate bounds
print(f"Latitude range:  {df_se['LATITUDE'].min():.4f} to {df_se['LATITUDE'].max():.4f}")
print(f"Longitude range: {df_se['LONGITUDE'].min():.4f} to {df_se['LONGITUDE'].max():.4f}")

Latitude range:  55.3379 to 68.4397
Longitude range: 11.0100 to 24.0186


## 7. Comparison with US Data Format

In [15]:
# Load sample US data
us_files = sorted(US_WP_DIR.glob("*.parquet"))
if us_files:
    df_us_sample = pd.read_parquet(us_files[0])
    print(f"US sample file: {us_files[0].name}")
    print(f"US sample records: {len(df_us_sample):,}")
    df_us_sample.head()

US sample file: 2024-03-04--data_01c29bf5-0107-dbc6-0042-fa070657a816_004_4_0.snappy.parquet
US sample records: 527,327


In [16]:
# Column comparison
if us_files:
    se_cols = set(df_se.columns)
    us_cols = set(df_us_sample.columns)
    
    print(f"Common columns: {len(se_cols & us_cols)}")
    print(f"\nSweden-only columns: {se_cols - us_cols}")
    print(f"\nUS-only columns: {us_cols - se_cols}")

Common columns: 18

Sweden-only columns: {'n_home_zones', 'unified_category', 'VISITOR_HOME_CBGS_WEIGHTED', 'PLACEKEY', 'cbgs_matches', 'WEIGHTED_VISIT_COUNTS', 'cbgs_total'}

US-only columns: {'VISITS_BY_EACH_HOUR', 'RELATED_SAME_DAY_BRAND', 'IS_DISTRIBUTOR', 'RELATED_SAME_WEEK_BRAND', 'PERSISTENT_ID', 'DEVICE_TYPE', 'BUCKETED_DWELL_TIMES', 'VISITOR_DAYTIME_CBGS', 'MEDIAN_DWELL', 'VISITOR_HOME_AGGREGATION', 'CLOSE_DATE', 'VISITS_BY_DAY', 'POI_CBG', 'VISITOR_COUNTRY_OF_ORIGIN', 'OPEN_DATE', 'PERSISTENT_ID_STORE', 'DISTANCE_FROM_HOME', 'TICKER', 'MSA_CODE', 'FOOTPRINT_ID'}


In [17]:
# Compare VISITOR_HOME_CBGS format
if us_files and 'VISITOR_HOME_CBGS' in df_us_sample.columns:
    us_cbgs_sample = df_us_sample['VISITOR_HOME_CBGS'].dropna().iloc[0]
    se_cbgs_sample = df_se['VISITOR_HOME_CBGS'].iloc[0]
    
    print("Sweden VISITOR_HOME_CBGS:")
    print(f"  Type: {type(se_cbgs_sample)}")
    print(f"  Sample: {str(se_cbgs_sample)[:150]}...")
    
    print("\nUS VISITOR_HOME_CBGS:")
    print(f"  Type: {type(us_cbgs_sample)}")
    print(f"  Sample: {str(us_cbgs_sample)[:150]}...")

Sweden VISITOR_HOME_CBGS:
  Type: <class 'str'>
  Sample: {"1281C1440": 1, "1480C1820": 5}...

US VISITOR_HOME_CBGS:
  Type: <class 'str'>
  Sample: {"391559312002":14}...


## 8. Data Quality Summary

In [18]:
# Coverage rates
coverage = pd.Series({
    'Location name': df_se['LOCATION_NAME'].notna().mean() * 100,
    'Coordinates': (df_se['LATITUDE'].notna() & df_se['LONGITUDE'].notna()).mean() * 100,
    'SUB_CATEGORY': df_se['SUB_CATEGORY'].notna().mean() * 100,
    'unified_category': df_se['unified_category'].notna().mean() * 100,
    'VISITOR_HOME_CBGS (non-empty)': (df_se['n_home_zones'] > 0).mean() * 100
}).round(1)

print("Coverage rates (%):\n")
print(coverage.to_string())

Coverage rates (%):

Location name                    100.0
Coordinates                      100.0
SUB_CATEGORY                      95.7
unified_category                  42.0
VISITOR_HOME_CBGS (non-empty)    100.0


In [19]:
# Summary statistics
print(f"""\nSummary Statistics:
==================
Records: {len(df_se):,}
Unique POIs: {df_se['PLACEKEY'].nunique():,}
Weeks: {df_se['DATE_RANGE_START'].nunique()}

Visits:
  Total visits: {df_se['VISIT_COUNTS'].sum():,.0f}
  Total weighted visits: {df_se['WEIGHTED_VISIT_COUNTS'].sum():,.0f}
  Mean visits/POI-week: {df_se['VISIT_COUNTS'].mean():.1f}
  Median visits/POI-week: {df_se['VISIT_COUNTS'].median():.1f}

Home zones:
  Mean zones/POI-week: {df_se['n_home_zones'].mean():.1f}
  Median zones/POI-week: {df_se['n_home_zones'].median():.1f}
  Total (POI, home_zone) pairs: ~{df_se['n_home_zones'].sum():,.0f}
""")


Summary Statistics:
Records: 995,666
Unique POIs: 349,311
Weeks: 6

Visits:
  Total visits: 6,621,210
  Total weighted visits: 24,367,881
  Mean visits/POI-week: 6.7
  Median visits/POI-week: 2.0

Home zones:
  Mean zones/POI-week: 2.8
  Median zones/POI-week: 2.0
  Total (POI, home_zone) pairs: ~2,773,925



## 9. Sample Records

In [20]:
# Top POIs by visits
top_pois = df_se.groupby('PLACEKEY').agg({
    'LOCATION_NAME': 'first',
    'CITY': 'first',
    'unified_category': 'first',
    'VISIT_COUNTS': 'sum',
    'n_home_zones': 'mean'
}).sort_values('VISIT_COUNTS', ascending=False)

top_pois.head(15)

,LOCATION_NAME,CITY,unified_category,VISIT_COUNTS,n_home_zones
PLACEKEY,,,,,
zzy-224@ayt-dn2-sdv,Air France,Stockholm-arlanda,NaN,7662,508.8
zzy-222@524-kqm-m49,Copenhagen Malmo Port,Malmo,NaN,5824,303.2
zzy-227@ayt-gmg-z75,Folktandvarden Stockholms lan,Kista,NaN,5351,423.6
zzy-224@az9-4gb-k4v,Linkoping City Airport AB,Linköping,NaN,4455,86.4
zzy-225@ayt-gjr-7h5,Mall of Scandinavia,Stockholm,NaN,4024,425.2
zzy-22b@ayt-gkp-f4v,Forlossningen Sodersjukhuset,Stockholm,NaN,3705,168.4
zzy-22y@ayt-gkd-dqf,Molins Fontan,Stockholm,entertainment_recreation,3595,332.6
zzy-222@52d-s4s-k9f,Port of Helsingborg,Helsingborg,NaN,3507,253.6
zzy-239@ayt-gnr-6p9,Bik Bok,Stockholm,retail_specialty,3140,216.8


In [21]:
# Sample records with high visitor diversity (many home zones)
diverse_pois = df_se.nlargest(10, 'n_home_zones')[[
    'PLACEKEY', 'LOCATION_NAME', 'CITY', 'unified_category', 
    'VISIT_COUNTS', 'n_home_zones', 'DATE_RANGE_START'
]]
diverse_pois

,PLACEKEY,LOCATION_NAME,CITY,unified_category,VISIT_COUNTS,n_home_zones,DATE_RANGE_START
926121,zzy-22y@ayt-gkd-dqf,Molins Fontan,Stockholm,entertainment_recreation,2328,906,2024-03-11
976040,zzy-247@52f-grv-x89,Lejontrappan,Gothenburg,entertainment_recreation,1834,818,2024-03-11
528121,zzy-224@ayt-dn2-sdv,Air France,Stockholm-arlanda,NaN,2056,695,2024-03-25
967056,zzy-23t@ayt-gkd-kfz,Jarla Trafikskola,Stockholm,NaN,1119,592,2024-03-18
851082,zzy-22g@52f-grq-j9z,Brf Umbra Kvillebacken,Göteborg,NaN,950,553,2024-03-18
731104,zzy-227@ayt-gmg-z75,Folktandvarden Stockholms lan,Kista,NaN,1361,529,2024-03-18
528120,zzy-224@ayt-dn2-sdv,Air France,Stockholm-arlanda,NaN,1731,525,2024-03-18
731105,zzy-227@ayt-gmg-z75,Folktandvarden Stockholms lan,Kista,NaN,1419,525,2024-03-25
967057,zzy-23t@ayt-gkd-kfz,Jarla Trafikskola,Stockholm,NaN,937,522,2024-03-25
528119,zzy-224@ayt-dn2-sdv,Air France,Stockholm-arlanda,NaN,1588,504,2024-03-11


In [22]:
# Examine a single POI's VISITOR_HOME_CBGS in detail
sample_poi = df_se.iloc[0]
print(f"POI: {sample_poi['LOCATION_NAME']}")
print(f"City: {sample_poi['CITY']}")
print(f"Category: {sample_poi['unified_category']}")
print(f"Visits: {sample_poi['VISIT_COUNTS']}")
print(f"\nVISITOR_HOME_CBGS (DeSO -> visitor count):")
cbgs = parse_cbgs(sample_poi['VISITOR_HOME_CBGS'])
for deso, count in sorted(cbgs.items(), key=lambda x: -x[1])[:10]:
    print(f"  {deso}: {count}")

POI: Vaderoarnas Vardshus & Konferens
City: Fjällbacka
Category: accommodation_travel
Visits: 6

VISITOR_HOME_CBGS (DeSO -> visitor count):
  1480C1820: 5
  1281C1440: 1
